# General workflow

## Preparation

In [ ]:
import os

import torch
from oocyte_decider.data_builder import OCTDataset, data_registration_main
from oocyte_decider.model_builder import CustomResNet50
from oocyte_explainability.visualize import imgify_blend
from zennit.attribution import Gradient
from zennit.composites import (
    EpsilonPlusFlat,
)
from zennit.core import RemovableHandleList
from zennit.image import imgify
from zennit.torchvision import ResNetCanonizer

from zennit_crp.attribution import ConditionalGradient
from zennit_crp.composites import MaskComposite, MultiComposite

### Paths

In [ ]:
data_registration_path = os.path.expandvars(r"$VSC_DATA") + "/data_registration"
models_path = os.path.expandvars(r"$VSC_DATA") + "/models"

### Model

In [ ]:
model = CustomResNet50()
state_dict = torch.load(
    os.path.join(models_path, "model_resnet50_50epochs.pth"),
    map_location=torch.device("cpu"),
)
model.load_state_dict(state_dict["model"])
model.eval()  # Set to evaluation mode

### Data

In [ ]:
train_df, test_df, train_groups, test_groups = data_registration_main(
    cache_dir=data_registration_path,
    cache_filename="processed_data.pkl",
)

train_data = OCTDataset(datamap=train_df)

In [ ]:
data_point = train_data[120]
sample = data_point["Sample"]
label = data_point["Label"]
label

In [ ]:
sample_img = imgify(sample[5], cmap="gray", vmin=0, vmax=1)
display(sample_img)

# add a batch dimension
data = sample.clone()[None]

In [ ]:
with torch.no_grad():
    output = model(data)
print(output)
print(output.argmax(1)[0].item())

## CRP

### Find the most relevant neurons in encoder.7.2.conv2

This is vanilla LRP where the relevances of the desired module are stored.

In [ ]:
# create a hook to keep track of intermediate outputs
def store_hook(module, input, output):
    # set the current module's attribute 'output' to the its tensor
    module.output = output
    # keep the output tensor gradient, even if it is not a leaf-tensor
    output.retain_grad()

In [ ]:
# use the ResNet-specific canonizer
canonizer = ResNetCanonizer()

# create a composite, specifying the canonizers
composite = EpsilonPlusFlat(canonizers=[canonizer])

# choose a target class for the attribution
target = torch.eye(4)[[label]]

# choose which modules to store the output of
modules_to_check = ["encoder.7.2.conv2"]

# create the attributor, specifying model and composite
with Gradient(model=model, composite=composite) as attributor:
    # register the store_hook AFTER the rule-hooks have been registered (by
    # entering the context) so we get the last output before the next module
    handles = RemovableHandleList(
        module.register_forward_hook(store_hook) for module_name, module in model.named_modules() if module_name in modules_to_check
    )

    # compute the model output and attribution
    output, attribution = attributor(data, target)

print("Prediction:", output.argmax(1)[0].item())
print("Individual predictions:", output.tolist())

# remove the hooks using store_hook
handles.remove()

In [ ]:
# obtain the relevance from the desired batch element
relevance = attribution[0]

# create an image of the visualize attribution
img = imgify(relevance, symmetric=True, cmap="coldnhot", grid=True)

# show the image
display(img)

In [ ]:
# overlay the relevance on the original image
img = imgify_blend(sample, relevance, grid=True)
display(img)

In [ ]:
# get encoder.7.2.conv2 relevance
sub_attribution = model.get_submodule("encoder.7.2.conv2").output.grad
sub_relevance = sub_attribution.sum((-2, -1))
sub_relevance /= (sub_relevance.abs().sum(-1, keepdim=True) + 1e-20)
rel_values, concept_ids = torch.topk(sub_relevance[0], 6)
print("Top concept IDs:", concept_ids)
print("Relevance values:", rel_values)

### Perform CRP for the most relevant neurons

In [ ]:
conditions = [{"encoder.7.2.conv2": [idx], "y": [0]} for idx in concept_ids]

for i, c in enumerate(conditions):
    print(c)

    # use the ResNet-specific canonizer
    canonizer = ResNetCanonizer()

    # create a composite, specifying the canonizers
    composite = EpsilonPlusFlat(canonizers=[canonizer])

    # determine the conditions for the layerwise relevance propagation
    conditions = [{"y": label}]

    # create a mask composite, specifying the conditions
    mask_composite = MaskComposite(conditions)

    # combine the composites, specifying the order of application
    # with mask_composite listed last, such that it is executed first (cf. backprop)
    final_composite = MultiComposite([composite, mask_composite])
    
    # create the attributor, specifying model and composite
    with ConditionalGradient(model=model, composite=composite) as attributor:
        # register the store_hook AFTER the rule-hooks have been registered (by
        # entering the context) so we get the last output before the next module
        handles = RemovableHandleList(
            module.register_forward_hook(store_hook) for module_name, module in model.named_modules() if module_name in conditions[0].keys()
        )

        # compute the model output and attribution
        output, attribution = attributor(data, target)

    print(f"Prediction: {output.argmax(1)[0].item()}")
    print(f"Individual predictions: {output.tolist()}")

    # remove the hooks using store_hook
    handles.remove()

    img = imgify_blend(sample, data.grad[0], grid=True)
    save_blend(
        img, img_num, label, c["encoder.7.2.conv2"][0], rel_values[i].item() * 100
    )